Load and split dataframe

In [1]:
import pandas as pd
import sys
sys.path.append("..")
from python_editor.data_processing import split_by_developer, get_vectorized_features_and_label

df = pd.read_pickle("../data/features_v2.pkl")
df = df.drop(columns=["embedding"])

train, test = split_by_developer(df, test_size=0.3, random_state=0)

Stack all features to get a vector for each data point

In [2]:
from python_editor.feature_generation_v2 import TRANSFORMED_FEATURES

X_train, y_train = get_vectorized_features_and_label(train, TRANSFORMED_FEATURES)
X_test, y_test = get_vectorized_features_and_label(test, TRANSFORMED_FEATURES)

Train a random-forest-regressor on generated features only. We use optuna for hyperparameter tuning and mlflow for experiment tracking

In [6]:
import optuna
import mlflow
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from python_editor.model_evaluation import get_metrics

mlflow.set_tracking_uri("sqlite:///../models/mlflow/mlflow.db")
mlflow.create_experiment("random_forest_regressor", artifact_location="../models/mlflow/artifacts")
mlflow.set_experiment("random_forest_regressor")

def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 50),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "oob_score": True,
        "random_state": 0,
        "n_jobs": -1
    }

    model = RandomForestRegressor(**params)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)

    # negative RMSE because sklearn maximizes score
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )

    rmse = -scores.mean()

    # Nested MLflow run for each trial
    with mlflow.start_run(nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("cv_rmse", rmse)

    return rmse


with mlflow.start_run(run_name="optuna_rf_search"):
    mlflow.set_tags(
        {
            "dataset": "features_v2.pkl",
            "processing": "data_processing.py",
            "feature_generation": "feature_generation_v2.py",
            "test_size": 0.3,
            "random_state": 0
        }
    )

    study = optuna.create_study(direction="minimize")

    study.optimize(objective, n_trials=50, show_progress_bar=True)

    best_params = study.best_params

    # Final model
    best_model = RandomForestRegressor(
        **best_params,
        oob_score=True,
        random_state=0,
        n_jobs=-1
    )

    best_model.fit(X_train, y_train)

    preds = best_model.predict(X_test)
    metrics = get_metrics(y_test, preds)

    # Log best results
    mlflow.log_params(best_params)
    mlflow.log_metric("best_cv_rmse", study.best_value)
    mlflow.log_metric("test_mae", metrics["MAE"])
    mlflow.log_metric("test_rmse", metrics["RMSE"])
    mlflow.log_metric("test_r2", metrics["R2"])

    mlflow.sklearn.log_model(best_model, "model")

print("Best Params:", best_params)
print(metrics)


[I 2026-04-25 23:36:39,528] A new study created in memory with name: no-name-5ce6e32b-3c76-4a0c-a4f0-ffac43eef5a4


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-04-25 23:36:44,553] Trial 0 finished with value: 2.5385625487012153 and parameters: {'n_estimators': 192, 'max_depth': 21, 'min_samples_split': 15, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 2.5385625487012153.
[I 2026-04-25 23:36:50,214] Trial 1 finished with value: 2.520260381111761 and parameters: {'n_estimators': 252, 'max_depth': 13, 'min_samples_split': 12, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 1 with value: 2.520260381111761.
[I 2026-04-25 23:36:54,368] Trial 2 finished with value: 2.5303546062116475 and parameters: {'n_estimators': 360, 'max_depth': 26, 'min_samples_split': 7, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 1 with value: 2.520260381111761.
[I 2026-04-25 23:36:55,204] Trial 3 finished with value: 2.529163553229511 and parameters: {'n_estimators': 118, 'max_depth': 48, 'min_samples_split': 20, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 1 with value: 2.520260381111761.

2026/04/25 23:38:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/25 23:38:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best Params: {'n_estimators': 455, 'max_depth': 49, 'min_samples_split': 3, 'min_samples_leaf': 2, 'max_features': 'log2'}
{'MAE': '2.030', 'RMSE': '2.584', 'R2': '0.375'}


We see that our third model is a little worse than the second one but we got rid of embbeddings burden which decreases latency and increases interpretability 

In [7]:
import pickle
from python_editor.model_evaluation import get_metrics

df_v2 = pd.read_pickle("../data/features_v2.pkl")
train_v2, test_v2 = split_by_developer(df_v2, test_size=0.3, random_state=0)
X_test_v2, y_test_v2 = get_vectorized_features_and_label(test_v2, TRANSFORMED_FEATURES)

with open("../models/model_v2.pkl", "rb") as f:
    model_v2 = pickle.load(f)


y_train_model_v2 = model_v2.oob_prediction_
y_train_model_v3 = best_model.oob_prediction_

y_test_model_v2 = model_v2.predict(X_test_v2)
y_test_model_v3 = best_model.predict(X_test)

print(f"Train v2:  {get_metrics(y_train, y_train_model_v2)}\nTest  v2:  {get_metrics(y_test_v2, y_test_model_v2)}")
print(f"Train v3:  {get_metrics(y_train, y_train_model_v3)}\nTest  v3:  {get_metrics(y_test, y_test_model_v3)}")

Train v2:  {'MAE': '1.925', 'RMSE': '2.422', 'R2': '0.372'}
Test  v2:  {'MAE': '1.975', 'RMSE': '2.463', 'R2': '0.432'}
Train v3:  {'MAE': '1.928', 'RMSE': '2.458', 'R2': '0.353'}
Test  v3:  {'MAE': '2.030', 'RMSE': '2.584', 'R2': '0.375'}


We save the new model

In [8]:
with open("../models/model_v3.pkl", "wb") as f:
    pickle.dump(best_model, f)

We create a function that takes text input, processes it, generate features and use the model to predict

In [ ]:
from python_editor.model_prediction import get_model_prediction_from_text
from python_editor.feature_generation_v2 import generate_transformed_features

row = pd.Series({"text": "import pandas as pd\nif True:\n\tpass"})

features, score = get_model_prediction_from_text(row, generate_transformed_features, best_model, embedding_dim=0)
features, score

2.563738462069574